In [ ]:
from time import perf_counter
import numpy as np
import openpyxl

In [2]:
# =========================
# CONFIG
# =========================
XLSX_PATH = "ANP_supermatrix.xlsx"
SHEET_NAME = "TEMPLATE_EMPTY"

DET_LABELS = ["DE1", "DE2"]
DIM_LABELS = ["D1", "D2", "D3", "D4", "D5", "D6"]

TOL = 1e-12
MAX_ITER = 20000
EXTRA_STABILITY_ITERS = 500

In [3]:
# =========================
# HELPERS
# =========================
def read_supermatrix_from_excel(path: str, sheet_name: str):
    wb = openpyxl.load_workbook(path, data_only=True)
    sh = wb[sheet_name]

    # Read column labels from row 1 starting at B1
    labels = []
    c = 2
    while True:
        v = sh.cell(row=1, column=c).value
        if v is None:
            break
        labels.append(str(v).strip())
        c += 1

    # Read row labels from col A starting at A2
    rows = []
    r = 2
    while True:
        v = sh.cell(row=r, column=1).value
        if v is None:
            break
        rows.append(str(v).strip())
        r += 1

    n = len(labels)
    if n == 0:
        raise ValueError("Tidak menemukan header kolom di baris 1 mulai B1.")
    if n != len(rows):
        raise ValueError(f"Ukuran label kolom ({n}) != label baris ({len(rows)}).")
    if labels != rows:
        raise ValueError("Urutan label kolom dan baris tidak identik. Samakan dulu (supermatrix harus square & konsisten).")

    M = np.zeros((n, n), dtype=float)
    for i in range(n):
        for j in range(n):
            val = sh.cell(row=2 + i, column=2 + j).value
            M[i, j] = float(val) if val is not None else 0.0

    return labels, M


def make_column_stochastic_with_self_loops(M: np.ndarray):
    """
    - If a column sum is 0: make it a self-loop (diagonal=1)
    - Then normalize columns to sum to 1
    """
    A = M.copy()
    col_sums = A.sum(axis=0)
    zero_cols = np.where(np.isclose(col_sums, 0.0))[0]

    for j in zero_cols:
        A[:, j] = 0.0
        A[j, j] = 1.0

    col_sums = A.sum(axis=0)
    # avoid division by zero (shouldn't happen after self-loops)
    if np.any(np.isclose(col_sums, 0.0)):
        raise ValueError("Masih ada kolom dengan jumlah 0 setelah self-loop. Cek matrix.")
    W = A / col_sums
    return W


def power_method_stationary(W: np.ndarray, tol=TOL, max_iter=MAX_ITER):
    """
    Find stationary distribution v such that v = W @ v (column-stochastic W).
    Start with uniform vector and iterate until convergence.
    """
    n = W.shape[0]
    v = np.ones(n) / n

    for it in range(max_iter):
        v_new = W @ v
        # normalize to avoid drift
        s = v_new.sum()
        if np.isclose(s, 0.0):
            raise ValueError("Vektor menjadi nol (sum=0). Cek jaringan/konektivitas.")
        v_new = v_new / s

        diff = np.linalg.norm(v_new - v, ord=1)
        v = v_new
        if diff < tol:
            return v, it + 1, diff

    return v, max_iter, diff


def stability_check(W: np.ndarray, v: np.ndarray, extra_iters=EXTRA_STABILITY_ITERS):
    v2 = v.copy()
    for _ in range(extra_iters):
        v2 = W @ v2
    v2 = v2 / v2.sum()
    return np.linalg.norm(v2 - v, ord=1)


def pick_indices(labels, wanted):
    idx = []
    for w in wanted:
        if w not in labels:
            raise ValueError(f"Label '{w}' tidak ditemukan.")
        idx.append(labels.index(w))
    return idx


def format_top(pairs, k=10):
    pairs_sorted = sorted(pairs, key=lambda x: x[1], reverse=True)
    return pairs_sorted[:k]


def normalize_within_cluster(weights: np.ndarray):
    s = weights.sum()
    if np.isclose(s, 0.0):
        return weights * 0.0
    return weights / s

In [4]:
# =========================
# MAIN (FAIR TIMING)
# =========================
timing = {}

t_total0 = perf_counter()

# --- 1) LOAD / PARSE (preprocessing) ---
t0 = perf_counter()
labels, M_full = read_supermatrix_from_excel(XLSX_PATH, SHEET_NAME)
timing["load_excel_parse"] = perf_counter() - t0

# Identify clusters
det_set = set(DET_LABELS)
dim_set = set(DIM_LABELS)
enb_labels = [lab for lab in labels if (lab not in det_set and lab not in dim_set)]

# --- 2) FULL NETWORK: build stochastic (prep) + limit iteration (core) ---
t0 = perf_counter()
W_full = make_column_stochastic_with_self_loops(M_full)
timing["prep_full_stochastic"] = perf_counter() - t0

t0 = perf_counter()
v_full, it_full, diff_full = power_method_stationary(W_full, tol=TOL, max_iter=MAX_ITER)
timing["core_full_limit"] = perf_counter() - t0

t0 = perf_counter()
stab_full = stability_check(W_full, v_full, extra_iters=EXTRA_STABILITY_ITERS)
timing["post_full_stability_check"] = perf_counter() - t0

# Extract ENB weights from full stationary vector
idx_enb = [labels.index(x) for x in enb_labels]
w_enb_raw = v_full[idx_enb]
w_enb_cluster = normalize_within_cluster(w_enb_raw)

# --- 3) SUBNETWORK (DE + DIM): build submatrix (prep) + limit iteration (core) ---
keep = DET_LABELS + DIM_LABELS
idx_keep = [labels.index(x) for x in keep]
M_sub = M_full[np.ix_(idx_keep, idx_keep)]

t0 = perf_counter()
W_sub = make_column_stochastic_with_self_loops(M_sub)
timing["prep_sub_stochastic"] = perf_counter() - t0

t0 = perf_counter()
v_sub, it_sub, diff_sub = power_method_stationary(W_sub, tol=TOL, max_iter=MAX_ITER)
timing["core_sub_limit"] = perf_counter() - t0

t0 = perf_counter()
stab_sub = stability_check(W_sub, v_sub, extra_iters=EXTRA_STABILITY_ITERS)
timing["post_sub_stability_check"] = perf_counter() - t0

# Extract DE & DIM weights from subnetwork stationary vector
idx_det_sub = [keep.index(x) for x in DET_LABELS]
idx_dim_sub = [keep.index(x) for x in DIM_LABELS]

w_det_raw = v_sub[idx_det_sub]
w_dim_raw = v_sub[idx_dim_sub]

w_det_cluster = normalize_within_cluster(w_det_raw)
w_dim_cluster = normalize_within_cluster(w_dim_raw)

timing["total_end_to_end"] = perf_counter() - t_total0


In [5]:
# =========================
# PRINT RESULTS
# =========================
core_total = timing["core_full_limit"] + timing["core_sub_limit"]
prep_total = timing["load_excel_parse"] + timing["prep_full_stochastic"] + timing["prep_sub_stochastic"]
post_total = timing["post_full_stability_check"] + timing["post_sub_stability_check"]

print("=== TIMING (seconds) ===")
print(f"Preprocessing (load+parse)           : {timing['load_excel_parse']:.6f}")
print(f"Prep (make stochastic) FULL          : {timing['prep_full_stochastic']:.6f}")
print(f"CORE (limit iteration) FULL          : {timing['core_full_limit']:.6f} | it={it_full}, diff={diff_full:.3e}")
print(f"Post (stability check) FULL          : {timing['post_full_stability_check']:.6f} | stab={stab_full:.3e}")
print(f"Prep (make stochastic) SUB (DE+DIM)  : {timing['prep_sub_stochastic']:.6f}")
print(f"CORE (limit iteration) SUB (DE+DIM)  : {timing['core_sub_limit']:.6f} | it={it_sub}, diff={diff_sub:.3e}")
print(f"Post (stability check) SUB           : {timing['post_sub_stability_check']:.6f} | stab={stab_sub:.3e}")
print()
print(f"CORE TOTAL (FULL + SUB)              : {core_total:.6f}")
print(f"END-TO-END TOTAL                     : {timing['total_end_to_end']:.6f}\n")

print("=== GLOBAL WEIGHTS (SUBNETWORK DE-DIM) ===")
print("Determinant (raw within subnetwork):")
for nm, w in zip(DET_LABELS, w_det_raw):
    print(f"  {nm}: {w:.10f}")
print("Determinant (normalized within DE cluster):")
for nm, w in zip(DET_LABELS, w_det_cluster):
    print(f"  {nm}: {w:.10f}")
print()

print("Dimension (raw within subnetwork):")
for nm, w in zip(DIM_LABELS, w_dim_raw):
    print(f"  {nm}: {w:.10f}")
print("Dimension (normalized within DIM cluster):")
for nm, w in zip(DIM_LABELS, w_dim_cluster):
    print(f"  {nm}: {w:.10f}")
print()

print("=== GLOBAL WEIGHTS (FULL NETWORK) ===")
print("Enabler (raw in full network):")
for nm, w in sorted(zip(enb_labels, w_enb_raw), key=lambda x: x[1], reverse=True):
    print(f"  {nm}: {w:.10f}")

print("\nEnabler (normalized within ENB cluster):")
for nm, w in sorted(zip(enb_labels, w_enb_cluster), key=lambda x: x[1], reverse=True):
    print(f"  {nm}: {w:.10f}")

print("\nTop 5 Enabler (cluster-normalized):")
for nm, w in format_top(list(zip(enb_labels, w_enb_cluster)), k=5):
    print(f"  {nm}: {w:.10f}")


=== TIMING (seconds) ===
Preprocessing (load+parse)           : 0.030056
Prep (make stochastic) FULL          : 0.003528
CORE (limit iteration) FULL          : 0.040105 | it=100, diff=7.995e-13
Post (stability check) FULL          : 0.002248 | stab=2.658e-12
Prep (make stochastic) SUB (DE+DIM)  : 0.000365
CORE (limit iteration) SUB (DE+DIM)  : 0.005120 | it=69, diff=7.757e-13
Post (stability check) SUB           : 0.001945 | stab=3.089e-13

CORE TOTAL (FULL + SUB)              : 0.045226
END-TO-END TOTAL                     : 0.085320

=== GLOBAL WEIGHTS (SUBNETWORK DE-DIM) ===
Determinant (raw within subnetwork):
  DE1: 0.3066384106
  DE2: 0.2951852870
Determinant (normalized within DE cluster):
  DE1: 0.5095153478
  DE2: 0.4904846522

Dimension (raw within subnetwork):
  D1: 0.0667476968
  D2: 0.0701305311
  D3: 0.0518552666
  D4: 0.0499576957
  D5: 0.0744621509
  D6: 0.0850229613
Dimension (normalized within DIM cluster):
  D1: 0.1676335241
  D2: 0.1761293444
  D3: 0.1302319256
  D4

## Uji Kompleksitas Empiris (Runtime & Memori)

Cell berikut menambahkan **uji skala** (empirical scaling) untuk proses inti ANP pada notebook ini, yaitu:
- **power_method_stationary** (iterasi limit / stationary vector)
- **stability_check** (iterasi tambahan untuk cek stabilitas)

Catatan:
- Ini **bukan** pembuktian Big-O secara matematis; ini *indikasi empiris* bagaimana waktu & memori berubah saat ukuran matriks (jumlah node) diperbesar/diperkecil.
- Kode inti di atas **tidak diubah**; cell ini hanya menjalankan benchmark tambahan.


In [6]:
import pandas as pd
import tracemalloc
from time import perf_counter

def _bench_once(func, *args, **kwargs):
    """Return (elapsed_seconds, peak_mem_mb, result)."""
    tracemalloc.start()
    t0 = perf_counter()
    res = func(*args, **kwargs)
    t1 = perf_counter()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return (t1 - t0), (peak / (1024**2)), res

def benchmark_anp_limit_scaling(W_base: np.ndarray, sizes, tol=TOL, max_iter=MAX_ITER, extra_iters=EXTRA_STABILITY_ITERS, trials=3):
    records = []
    nmax = W_base.shape[0]
    sizes = [int(s) for s in sizes if int(s) >= 2 and int(s) <= nmax]
    for n in sizes:
        W = W_base[:n, :n].copy()

        # Pastikan stochastic (biar adil & stabil). Menggunakan helper yang sudah ada.
        try:
            W = make_column_stochastic(W)
        except NameError:
            # fallback: kolom dinormalisasi manual
            colsum = W.sum(axis=0, keepdims=True)
            colsum[colsum == 0] = 1.0
            W = W / colsum

        times_pm, mems_pm = [], []
        times_st, mems_st = [], []

        it_last, diff_last, stab_last = None, None, None

        for _ in range(trials):
            # 1) power method (stationary vector)
            dt, pm_mb, (v, it, diff) = _bench_once(power_method_stationary, W, tol=tol, max_iter=max_iter)
            times_pm.append(dt); mems_pm.append(pm_mb)
            it_last, diff_last = it, diff

            # 2) stability check (extra iters)
            dt2, st_mb, stab = _bench_once(stability_check, W, v, extra_iters=extra_iters)
            times_st.append(dt2); mems_st.append(st_mb)
            stab_last = stab

        records.append({
            "n_node": n,
            "pm_median_s": float(np.median(times_pm)),
            "pm_peak_mem_mb": float(np.max(mems_pm)),
            "stability_median_s": float(np.median(times_st)),
            "stability_peak_mem_mb": float(np.max(mems_st)),
            "pm_it": int(it_last) if it_last is not None else None,
            "pm_diff": float(diff_last) if diff_last is not None else None,
            "stability_score": float(stab_last) if stab_last is not None else None,
        })
    return pd.DataFrame(records)

# ---------- Jalankan benchmark ----------
# Pilih matriks yang ingin diuji: FULL atau SUB
W_target = W_full  # ganti ke W_sub kalau mau

N = W_target.shape[0]
sizes = sorted(set([min(N, s) for s in [5, 10, 15, 20, 25, 30, 40, 50, 75, 100, N] if s <= N]))

bench_df = benchmark_anp_limit_scaling(W_target, sizes=sizes, trials=3)
bench_df


,n_node,pm_median_s,pm_peak_mem_mb,stability_median_s,stability_peak_mem_mb,pm_it,pm_diff,stability_score
0,5,0.021290,0.004810,0.002639,0.001396,85,9.467982e-13,3.956419e-13
1,10,0.059844,0.002201,0.001983,0.001511,403,9.673112e-13,1.488757e-11
2,15,0.019016,0.002720,0.002330,0.001625,142,9.215432e-13,4.570988e-12
3,20,0.012055,0.002426,0.001993,0.001740,100,7.998504e-13,2.657903e-12
